> # SOLUTIONS — Block 3 — Template A
>
> The participant notebook is `block_3_—_template_a.ipynb`. Differences from this version are limited to (a) blanks filled in, (b) discussion answers added inline, and (c) this banner.

---


# Block 3 — Template A: The Process Spec

**ODSC Tutorial: Spec-Driven Simulation Modeling**

Block 2 produced the distributions and parameters from data. Now we put them into a structured spec — **Template A** — that captures everything the simulation needs to know *before any code is written*.

Three things make Template A worth the effort:

1. **It forces clarity.** Modeling decisions surface during the spec-writing, where they cost minutes to fix, not after the simulation runs and produces wrong-but-plausible answers.
2. **It is reviewable by domain experts.** The shop owner can read this and tell you whether you got the system right. You don't ask them to read SimPy.
3. **It becomes the LLM prompt.** Block 4's code generation feeds the *complete* Template A into the prompt. A vague spec produces vague code. A precise spec produces tight, reviewable code on the first try.

In this block we work through Template A for the coffee shop, with four strategic blanks for you to fill in. Filling them requires *modeling decisions*, not lookups — that is the point.

## Setup

Just imports and a small helper for clean output. No SimPy yet — Template A is documentation, not code.

In [1]:
import textwrap
from typing import List, Tuple

# Sentinel for unfilled blanks. Lets the notebook execute top-to-bottom
# even before participants fill anything in — the validation cell at the
# end detects the sentinel and reports which blanks are still empty.
___ = "???"

# A tiny helper that renders our spec values inline cleanly
def show(label: str, value, indent: int = 2) -> None:
    pad = " " * indent
    print(f"{pad}{label:<32} {value}")

## Part 1 — Read the spec sections that are already filled in

The first three sections of Template A are filled in for you. Read them carefully — they're the input you need to make the right decisions in Part 2.

### Section 1 — Problem statement

> **Business question:** Should the coffee shop hire a third barista during the morning rush, or is there a cheaper way to fix the wait-time problem?
>
> **Decision:** A staffing or process change that brings the worst-customer wait to under five minutes during the rush, while keeping daily labor cost under \$350.
>
> **Primary KPI:** P90 wait time (target: < 5 minutes during the morning rush).
>
> **Secondary metrics:** Walkaway rate, mean wait, per-barista utilization.
>
> **Cost metric:** Daily labor cost = barista-hours × \$17/hour.

### Section 2 — Entities

> **Entity type:** Customer.
>
> **Attributes per customer:**
> - `arrival_time` — when they joined the line, in minutes from store open.
> - `order_type` — one of `drip`, `espresso`, `blended`.
> - `outcome` — `served` if served, `walkaway` if they balked at the door.
>
> **Lifecycle:** arrive → check the line → either balk and leave, or join the queue → wait → be served → depart.

### Section 3 — Resources

> **Resource type:** Barista.
>
> **Capacity:** Number of baristas on shift. Varies by time window (this is what staffing scenarios change).
>
> **Service rule:** First-in, first-out within the queue. Each barista handles a customer end-to-end (order taking + drink prep, no hand-off).
>
> **Service time per customer:** drawn from a lognormal distribution that depends on `order_type` (Section 4).

## Part 2 — Fill in the strategic blanks

Four blanks. Each requires you to think through the modeling decision, not look it up.

### ▶ Blank 1: Espresso service-time distribution

We fit lognormal distributions per order type in Block 2. Drip and blended are filled in below. Espresso is missing.

**Hint.** Espresso sits between drip and blended in real-space mean (around three and a half minutes) but has *more variability* than either — milk steaming and machine warming introduce inconsistency that drip doesn't have and that blended drinks have already absorbed into their longer baseline. Pick `mu` and `sigma` accordingly. Two decimal places is fine.

In [2]:
# Lognormal parameters (mu, sigma) in log-space.
# Real-space mean = exp(mu + sigma**2 / 2).
order_types = {
    "drip":     {"prob": 0.30, "mu": 0.70, "sigma": 0.30},
    "espresso": {"prob": 0.50, "mu": 1.20, "sigma": 0.35},   # ◀ FILL THESE IN
    "blended":  {"prob": 0.20, "mu": 1.50, "sigma": 0.30},
}

# Quick sanity print: weighted mean service time should be around 3.34 min
# (the number we recovered from data in Block 2)
import math

def real_mean(p):
    if isinstance(p["mu"], (int, float)) and isinstance(p["sigma"], (int, float)):
        return math.exp(p["mu"] + 0.5 * p["sigma"]**2)
    return None

try:
    means = [real_mean(p) for p in order_types.values()]
    if any(m is None for m in means):
        print("Espresso parameters not yet filled in.")
    else:
        weighted = sum(p["prob"] * m for p, m in zip(order_types.values(), means))
        print(f"Weighted mean service time: {weighted:.2f} min  (target ≈ 3.34)")
except Exception as e:
    print(f"Could not compute weighted mean: {type(e).__name__}: {e}")

Weighted mean service time: 3.33 min  (target ≈ 3.34)


### ▶ Blank 2: Balking threshold

The shop owner says customers walk away when the line gets too long. The simulation needs to know *how long is too long* — a queue length that triggers balking.

**Hint.** Look at the physical layout. The shop's entryway can hold roughly six to eight people standing in line before the line spills out the door or blocks the registers. Pick the threshold that captures the customer behavior the owner described.

In [3]:
# A customer arriving to a queue of >= balking_threshold turns around and leaves.
balking_threshold = 8   # ◀ FILL IN  (integer)

### ▶ Blank 3: Scenario A staffing schedule

The shop runs three time windows. Baseline staffing is given; you fill in Scenario A.

| Window | Time | Baseline baristas | Scenario A baristas |
|--------|------|-------------------|---------------------|
| Morning rush | 7:00–9:00 AM (0–120 min) | 2 | ? |
| Midday | 9:00 AM–2:00 PM (120–420 min) | 2 | ? |
| Afternoon | 2:00–5:00 PM (420–600 min) | 1 | ? |

**Hint.** Scenario A is called "Rush boost" — it adds capacity *only where the bottleneck is*. Block 2's analysis showed the morning rush at 0.83 utilization (knee of the curve) and the rest of the day under 0.6. Make the schedule reflect that.

In [4]:
# Each tuple: (start_minute, end_minute, n_baristas).
# Minutes are measured from store open at 7 AM.

baseline_staffing = [
    (0,   120, 2),   # Morning rush
    (120, 420, 2),   # Midday
    (420, 600, 1),   # Afternoon
]

# Fill in Scenario A's three windows below.
scenario_a_staffing = [
    (0,   120, 3),   # Morning rush     ◀ filled: +1 barista during the rush
    (120, 420, 2),   # Midday           ◀ filled: same as baseline
    (420, 600, 1),   # Afternoon        ◀ filled: same as baseline
]

### ▶ Blank 4: Scenario A daily labor cost

Once the schedule is set, the cost is mechanical: sum of (baristas × hours × hourly wage). Hourly wage is \$17.

**Hint.** Compute it from `scenario_a_staffing` rather than typing a number in. That way if you change the schedule, the cost updates automatically — and that pattern is exactly what production simulation code should look like.

In [5]:
HOURLY_WAGE = 17  # $/hour per barista

def daily_cost(schedule: List[Tuple[int, int, int]]) -> float:
    """Sum of baristas × hours × wage across all windows in the schedule."""
    total = 0.0
    for start_min, end_min, n_baristas in schedule:
        hours = (end_min - start_min) / 60.0
        total += n_baristas * hours * HOURLY_WAGE
    return total

# Compute and report
try:
    baseline_cost   = daily_cost(baseline_staffing)
    scenario_a_cost = daily_cost(scenario_a_staffing)
    show("Baseline daily cost",   f"${baseline_cost:.0f}")
    show("Scenario A daily cost", f"${scenario_a_cost:.0f}")
    show("Scenario A vs baseline", f"${scenario_a_cost - baseline_cost:.0f}/day extra")
except Exception as e:
    print(f"Schedule not complete or daily_cost not implemented: {e}")

  Baseline daily cost              $289
  Scenario A daily cost            $323
  Scenario A vs baseline           $34/day extra


## Part 3 — Validate your spec

The cell below checks each blank. It tells you which ones look right, which look off, and which haven't been filled in. Don't worry about getting everything right on the first run — read the feedback, adjust, re-run.

In [6]:
def is_unfilled(value) -> bool:
    """Return True if a blank looks unfilled (still the ___ sentinel or None)."""
    if value is None:
        return True
    if isinstance(value, str):
        if value == "???" or "_" in value or "FILL" in value.upper():
            return True
    return False


def check_espresso() -> Tuple[bool, str]:
    p = order_types.get("espresso", {})
    mu, sigma = p.get("mu"), p.get("sigma")
    if is_unfilled(mu) or is_unfilled(sigma):
        return False, "Espresso parameters not filled in."
    # Reasonable range checks
    if not (0.9 <= mu <= 1.4):
        return False, f"mu={mu} is outside the reasonable range. Espresso real-space mean should be 2.5–4 min."
    if not (0.25 <= sigma <= 0.45):
        return False, f"sigma={sigma} is outside the reasonable range. Espresso variability should be slightly higher than drip and blended."
    real_mean_val = math.exp(mu + 0.5 * sigma**2)
    if not (3.0 <= real_mean_val <= 4.0):
        return False, f"Real-space mean works out to {real_mean_val:.2f} min — outside 3.0–4.0."
    return True, f"mu={mu}, sigma={sigma}  →  real-space mean ≈ {real_mean_val:.2f} min."


def check_balking() -> Tuple[bool, str]:
    bt = globals().get("balking_threshold")
    if is_unfilled(bt):
        return False, "balking_threshold not filled in."
    if not isinstance(bt, int):
        return False, f"balking_threshold should be an integer, got {type(bt).__name__}."
    if not (5 <= bt <= 12):
        return False, f"balking_threshold = {bt}. The owner described 6–8 people as the trigger; pick something in that range."
    return True, f"balking_threshold = {bt}."


def check_staffing() -> Tuple[bool, str]:
    sched = globals().get("scenario_a_staffing")
    if not sched or any(is_unfilled(n) for _, _, n in sched):
        return False, "Scenario A staffing not fully filled in."
    if len(sched) != 3:
        return False, f"Expected 3 windows, got {len(sched)}."
    morning, midday, afternoon = sched[0][2], sched[1][2], sched[2][2]
    if morning <= 2:
        return False, (f"Morning has {morning} baristas, same as or below baseline (2). "
                       f"Scenario A is supposed to *add* capacity during the rush.")
    if midday != 2 or afternoon != 1:
        return False, (f"Midday/afternoon should match baseline (2 and 1) for the *rush boost* scenario. "
                       f"Got midday={midday}, afternoon={afternoon}.")
    return True, f"Scenario A: {morning}/{midday}/{afternoon} baristas across rush/midday/afternoon."


def check_cost() -> Tuple[bool, str]:
    try:
        cost = daily_cost(scenario_a_staffing)
    except Exception as e:
        return False, f"daily_cost(...) raised {type(e).__name__}: {e}. Implement the function body."
    if cost is None or cost == 0.0:
        return False, "daily_cost returned 0 — looks like the function body wasn't implemented."
    if not (315 <= cost <= 330):
        return False, f"Scenario A cost = ${cost:.0f}. Expected around $323. Check the formula and units (minutes vs hours)."
    return True, f"Scenario A daily cost = ${cost:.0f}  ($34 above baseline)."


checks = [
    ("1. Espresso distribution",  check_espresso),
    ("2. Balking threshold",      check_balking),
    ("3. Scenario A staffing",    check_staffing),
    ("4. Scenario A daily cost",  check_cost),
]

print("Validating your Template A blanks")
print("=" * 70)
n_passed = 0
for label, fn in checks:
    try:
        ok, msg = fn()
    except Exception as e:
        ok, msg = False, f"check raised {type(e).__name__}: {e}"
    status = "✓ pass" if ok else "✗ fix"
    print(f"  [{status}]  {label}")
    print(f"           {msg}")
    if ok:
        n_passed += 1

print()
print(f"Passed: {n_passed} of {len(checks)}.")
if n_passed == len(checks):
    print("\nTemplate A is complete. Moving on to Block 4.")
else:
    print("\nKeep going — adjust the blanks above and re-run this cell.")

Validating your Template A blanks
  [✓ pass]  1. Espresso distribution
           mu=1.2, sigma=0.35  →  real-space mean ≈ 3.53 min.
  [✓ pass]  2. Balking threshold
           balking_threshold = 8.
  [✓ pass]  3. Scenario A staffing
           Scenario A: 3/2/1 baristas across rush/midday/afternoon.
  [✓ pass]  4. Scenario A daily cost
           Scenario A daily cost = $323  ($34 above baseline).

Passed: 4 of 4.

Template A is complete. Moving on to Block 4.


## What this spec becomes in Block 4

The complete Template A is the *input* to Block 4's code-generation prompt. The full Template B prompt looks like this:

```
You are an expert simulation engineer. Generate a SimPy 4.x implementation
of the discrete event simulation specified below. Use this five-function
architecture:

  - entity_process(env, name, baristas, scenario, rng, metrics)
  - entity_generator(env, baristas, scenario, rng, metrics)
  - run_single_replication(scenario, seed)
  - run_scenario(scenario, n_replications)
  - compare_scenarios(results)

# SPECIFICATION

[the four sections you just read, plus the four blanks you just filled in]
```

The numbers you put into the blanks above — the espresso `mu` and `sigma`, the balking threshold of 8, the Scenario A schedule of 3/2/1, the cost of \$323 — are the same numbers that show up in the generated SimPy code. **Spec is prompt is code.** Garbage in here is garbage in there.

In **Block 4** we run the code that this spec produces.

## What we leave Block 3 with

A complete Template A. Every section filled in, every parameter cited to its source, every scenario costed out. A document the shop owner could read in five minutes and tell you whether you got the system right.

In Block 4 we hand it to an LLM, get SimPy code back, and run the baseline.